# AICL on the Natural Questions (NQ) Dataset

This notebook demonstrates `AICLContextSelector` (Adaptive In-Context Learning) applied to the
real NQ dataset, using the `pyterrier/ragwiki-terrier` BM25 index.

Reference: Chandra, M., Ganguly, D., Ounis, I. (2025). *One Size Doesn't Fit All: Predicting the
Number of Examples for In-Context Learning*. ECIR 2025. https://arxiv.org/abs/2403.06402

Pipeline: load NQ → retrieve with BM25 (fixed k=10) → fit AICL to predict per-query k →
compare fixed-k vs. adaptive-k context sizes.

## 1. Setup

In [ ]:
!pip install -q python-terrier pyterrier-rag pyterrier-alpha scikit-learn

## 2. Load the NQ dataset

In [ ]:
import pyterrier as pt
import pandas as pd
from pyterrier_rag.aicl import AICLContextSelector

dataset = pt.get_dataset('rag:nq')

train_topics = dataset.get_topics('train').head(500)
train_answers = dataset.get_answers('train')
answers_sample = train_answers[train_answers['qid'].isin(train_topics['qid'])]

test_topics = dataset.get_topics('test').head(200)
test_answers = dataset.get_answers('test')
test_answers_sample = test_answers[test_answers['qid'].isin(test_topics['qid'])]

print(f'Train queries: {len(train_topics)}')
print(f'Test queries:  {len(test_topics)}')

## 3. Load the Wikipedia BM25 index

In [ ]:
sparse_index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')
bm25 = sparse_index.bm25(include_fields=['docno', 'text', 'title'])
print('Index ready.')

## 4. Retrieve documents (fixed k=10)

In [ ]:
train_results = (bm25 % 10).transform(train_topics)
test_results = (bm25 % 10).transform(test_topics)

print(f'Train results shape: {train_results.shape}')
print(f'Test results shape:  {test_results.shape}')

## 5. Build training labels and fit AICL

In [ ]:
labels = AICLContextSelector.build_labels_from_results(
    train_results, answers_sample, answer_col='answers', k_max=10
)
print(f'Labels built for {len(labels)} queries')

aicl = AICLContextSelector(k_max=10)
aicl.fit(train_results, labels)
print('AICL fitted.')

## 6. Apply AICL to the test set

In [ ]:
test_filtered = aicl.transform(test_results)

avg_before = len(test_results) / len(test_topics)
avg_after = len(test_filtered) / len(test_topics)

print(f'Average docs per query BEFORE AICL: {avg_before:.1f}')
print(f'Average docs per query AFTER AICL:  {avg_after:.1f}')
print(f'Context reduction: {100 * (1 - avg_after / avg_before):.1f}%')

k_preds = aicl.predict_k(test_results)
k_values = list(k_preds.values())
print('\nPredicted k distribution:')
print(pd.Series(k_values).value_counts().sort_index())

## 7. Results summary

In [ ]:
import numpy as np

label_arr = np.array(labels)

print('=' * 50)
print('AICL RESULTS SUMMARY ON NQ DATASET')
print('=' * 50)
print(f'Training queries: {len(train_topics)}')
print(f'Test queries:     {len(test_topics)}')
print(f'Max k:            10')
print()
print(f'BEFORE AICL: fixed k=10 docs per query')
print(f'AFTER AICL:  avg {avg_after:.1f} docs per query')
print(f'Context reduction: {100 * (1 - avg_after / avg_before):.1f}%')
print()
print('Answer found in top-k docs (train set):')
for k in range(10):
    print(f'  k={k+1}: {label_arr[:, k].mean() * 100:.1f}%')
print()
print('AICL adapts context size per query, giving smaller context to')
print('queries that need fewer documents and reducing noise for the LLM reader.')